# Data Exploration: Psychiatric Gene Expression

## Project objective
Investigate whether machine-learning models can distinguish schizophrenia, bipolar disorder, and healthy controls using brain gene-expression profiles.

## Dataset
- **Accession:** GSE62191
- **Source:** NCBI Gene Expression Omnibus (GEO)
- **Task:** Three-class classification

## Objectives
1. Download and inspect the dataset using GEOparse.
2. Audit sample labels, dimensions, missing values, and data quality.
3. Prepare the data for machine learning.
4. Establish a baseline classifier and evaluate its performance.

## Download GEO Dataset

In [7]:
from pathlib import Path
import requests
import GEOparse

PROJECT_DIR = Path.cwd().parent
GEO_DIR = PROJECT_DIR / "data" / "geo"
GEO_DIR.mkdir(parents=True, exist_ok=True)

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE62nnn/GSE62191/soft/GSE62191_family.soft.gz"

soft_file = GEO_DIR / "GSE62191_family.soft.gz"
temp_file = GEO_DIR / "GSE62191_family.soft.gz.part"

with requests.get(url, stream=True, timeout=(20, 120)) as response:
    response.raise_for_status()
    with open(temp_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

temp_file.replace(soft_file)

print("Download complete:", soft_file)

Download complete: C:\Users\taraj\OneDrive\Documents\GitHub\psychiatric-gene-expression-ml\data\geo\GSE62191_family.soft.gz
File size (MB): 28.46


## Load GEO dataset

In [28]:
gse = GEOparse.get_GEO(filepath=str(soft_file), destdir=str(GEO_DIR))

print("Study:", gse.name)
print("Number of samples:", len(gse.gsms))
print("Platforms:", list(gse.gpls.keys()))

12-Oct-2026 20:07:26 INFO GEOparse - Parsing C:\Users\taraj\OneDrive\Documents\GitHub\psychiatric-gene-expression-ml\data\geo\GSE62191_family.soft.gz: 
12-Oct-2026 20:07:26 DEBUG GEOparse - DATABASE: GeoMiame
12-Oct-2026 20:07:26 DEBUG GEOparse - SERIES: GSE62191
12-Oct-2026 20:07:26 DEBUG GEOparse - PLATFORM: GPL4133
12-Oct-2026 20:07:27 WARNING GEOTypes - Detected duplicated columns in d GPL4133. Correcting.

12-Oct-2026 20:07:27 DEBUG GEOparse - SAMPLE: GSM1521625
12-Oct-2026 20:07:27 DEBUG GEOparse - SAMPLE: GSM1521626
12-Oct-2026 20:07:27 DEBUG GEOparse - SAMPLE: GSM1521627
12-Oct-2026 20:07:27 DEBUG GEOparse - SAMPLE: GSM1521628
12-Oct-2026 20:07:27 DEBUG GEOparse - SAMPLE: GSM1521629
12-Oct-2026 20:07:28 DEBUG GEOparse - SAMPLE: GSM1521630
12-Oct-2026 20:07:28 DEBUG GEOparse - SAMPLE: GSM1521631
12-Oct-2026 20:07:28 DEBUG GEOparse - SAMPLE: GSM1521632
12-Oct-2026 20:07:28 DEBUG GEOparse - SAMPLE: GSM1521633
12-Oct-2026 20:07:28 DEBUG GEOparse - SAMPLE: GSM1521634
12-Oct-2026 20:

Study: GSE62191
Number of samples: 88
Platforms: ['GPL4133']


## Inspect the metadata for the first sample

In [9]:
sample_id, sample = next(iter(gse.gsms.items()))

print("Sample ID:", sample_id)
print("\nMetadata fields:")
for key, value in sample.metadata.items():
    print(f"{key}: {value}")

Sample ID: GSM1521625

Metadata fields:
title: ['BP1']
geo_accession: ['GSM1521625']
status: ['Public on Oct 11 2014']
submission_date: ['Oct 08 2014']
last_update_date: ['Oct 14 2014']
type: ['RNA']
channel_count: ['2']
source_name_ch1: ['frontal cortex, bipolar disorder']
organism_ch1: ['Homo sapiens']
taxid_ch1: ['9606']
characteristics_ch1: ['tissue: brain (frontal cortex)', 'disease state: bipolar disorder', 'age: 29 yr', 'population: white', 'dsm-iv: 296.54', 'age of onset: 22 yr']
treatment_protocol_ch1: ['N/A']
growth_protocol_ch1: ['Frozen tissue.']
molecule_ch1: ['total RNA']
extract_protocol_ch1: ["Total RNA was isolated using TRIzol (Sigma) according to the manufacturer's instructions.", 'RNA concentration was determined by spectrophotometry (NanoDrop, Thermo Scientific, US), and integrity was assessed using the 2100 Bioanalyzer (Agilent Technologies, Waldbronn, Germany).']
label_ch1: ['Cy3']
label_protocol_ch1: ["Agilent Technologies' manufacturer procedures."]
source_name

## Inspect head of expression features for first sample

In [10]:
print("Sample data table shape:", sample.table.shape)
display(sample.table.head())

Sample data table shape: (30635, 2)


,ID_REF,VALUE
0,12,1.428326
1,13,1.442081
2,14,2.670825
3,15,1.467611
4,16,3.858828


## Isolate relevant metadata

In [13]:
import pandas as pd

sample_records = []

for sample_id, gsm in gse.gsms.items():
    sample_records.append({
        "sample_id": sample_id,
        "title": gsm.metadata.get("title", [""])[0],
        "characteristics": gsm.metadata.get(
            "characteristics_ch1", []
        )
    })

metadata_df = pd.DataFrame(sample_records)

display(metadata_df.head(10))
print("Total samples:", len(metadata_df))

,sample_id,title,characteristics
0,GSM1521625,BP1,"[tissue: brain (frontal cortex), disease state..."
1,GSM1521626,BP2,"[tissue: brain (frontal cortex), disease state..."
2,GSM1521627,BP3,"[tissue: brain (frontal cortex), disease state..."
3,GSM1521628,BP4,"[tissue: brain (frontal cortex), disease state..."
4,GSM1521629,BP5,"[tissue: brain (frontal cortex), disease state..."
5,GSM1521630,BP6,"[tissue: brain (frontal cortex), disease state..."
6,GSM1521631,BP7,"[tissue: brain (frontal cortex), disease state..."
7,GSM1521632,BP8,"[tissue: brain (frontal cortex), disease state..."
8,GSM1521633,BP9,"[tissue: brain (frontal cortex), disease state..."
9,GSM1521634,BP10,"[tissue: brain (frontal cortex), disease state..."


Total samples: 88


## Convert the characteristics list into separate columns

In [14]:
def parse_characteristics(characteristics):
    parsed = {}

    for item in characteristics:
        if ":" in item:
            key, value = item.split(":", 1)
            parsed[key.strip()] = value.strip()

    return parsed


characteristics_df = metadata_df["characteristics"].apply(
    parse_characteristics
).apply(pd.Series)

# Combine sample ID, title, and the newly separated fields
metadata_clean = pd.concat(
    [
        metadata_df[["sample_id", "title"]].reset_index(drop=True),
        characteristics_df.reset_index(drop=True)
    ],
    axis=1
)

display(metadata_clean.head(10))
print("Columns:", metadata_clean.columns.tolist())

,sample_id,title,tissue,disease state,age,population,dsm-iv,age of onset,gender
0,GSM1521625,BP1,brain (frontal cortex),bipolar disorder,29 yr,white,296.54,22 yr,NaN
1,GSM1521626,BP2,brain (frontal cortex),bipolar disorder,58 yr,white,296.54,27 yr,NaN
2,GSM1521627,BP3,brain (frontal cortex),bipolar disorder,54 yr,white,296.89,45 yr,NaN
3,GSM1521628,BP4,brain (frontal cortex),bipolar disorder,42 yr,white,296.64,20 yr,NaN
4,GSM1521629,BP5,brain (frontal cortex),bipolar disorder,63 yr,white,296.89,43 yr,NaN
5,GSM1521630,BP6,brain (frontal cortex),bipolar disorder,64 yr,white,295.7,19 yr,NaN
6,GSM1521631,BP7,brain (frontal cortex),bipolar disorder,59 yr,white,296.53,25 yr,NaN
7,GSM1521632,BP8,brain (frontal cortex),bipolar disorder,51 yr,white,296.44,23 yr,NaN
8,GSM1521633,BP9,brain (frontal cortex),bipolar disorder,49 yr,white,296.54,20 yr,NaN
9,GSM1521634,BP10,brain (frontal cortex),bipolar disorder,41 yr,white,296.72,14 yr,NaN


Columns: ['sample_id', 'title', 'tissue', 'disease state', 'age', 'population', 'dsm-iv', 'age of onset', 'gender']


## Inspect missing values in Gender Column

In [16]:
print("Gender values:")
print(metadata_clean["gender"].value_counts(dropna=False))

Gender values:
gender
NaN     87
male     1
Name: count, dtype: int64


## Inspect disease state column

In [17]:
print("Diagnosis counts:")
print(metadata_clean["disease state"].value_counts(dropna=False))

print("\nUnique diagnosis labels:")
print(metadata_clean["disease state"].unique())

Diagnosis counts:
disease state
healthy control     30
bipolar disorder    29
schizophrenia       29
Name: count, dtype: int64

Unique diagnosis labels:
['bipolar disorder' 'healthy control' 'schizophrenia']


## Combine expression measurements from samples into one table

In [18]:
expression_data = {}

for sample_id, gsm in gse.gsms.items():
    expression_data[sample_id] = gsm.table.set_index("ID_REF")["VALUE"]

X = pd.DataFrame(expression_data).T

# Make sample IDs the index and ensure the features are numeric
X.index.name = "sample_id"
X = X.apply(pd.to_numeric, errors="coerce")

print("Expression matrix shape:", X.shape)
display(X.iloc[:5, :5])

Expression matrix shape: (88, 30635)


ID_REF,12,13,14,15,16
sample_id,,,,,
GSM1521625,1.428326,1.442081,2.670825,1.467611,3.858828
GSM1521626,1.349548,0.969723,2.612003,1.147206,3.869557
GSM1521627,1.213370,1.782278,2.757108,1.183939,3.766497
GSM1521628,1.361207,1.365008,2.460249,1.372995,3.520331
GSM1521629,1.363634,1.416774,2.515373,1.368402,3.817245


## Inspect probe annotation table (for biological interpretation later)

In [20]:
gpl = gse.gpls["GPL4133"]

print("Annotation table shape:", gpl.table.shape)
print("Annotation columns:")
print(gpl.table.columns.tolist())

display(gpl.table.head())

Annotation table shape: (45220, 22)
Annotation columns:
['ID', 'COL', 'ROW', 'NAME', 'SPOT_ID', 'CONTROL_TYPE', 'REFSEQ', 'GB_ACC', 'GENE', 'GENE_SYMBOL', 'GENE_NAME', 'UNIGENE_ID', 'ENSEMBL_ID', 'TIGR_ID', 'ACCESSION_STRING', 'CHROMOSOMAL_LOCATION', 'CYTOBAND', 'DESCRIPTION', 'GO_ID', 'SEQUENCE', 'SPOT_ID.1', 'ORDER']


,ID,COL,ROW,NAME,SPOT_ID,CONTROL_TYPE,REFSEQ,GB_ACC,GENE,GENE_SYMBOL,...,ENSEMBL_ID,TIGR_ID,ACCESSION_STRING,CHROMOSOMAL_LOCATION,CYTOBAND,DESCRIPTION,GO_ID,SEQUENCE,SPOT_ID.1,ORDER
0,1,266,170,GE_BrightCorner,GE_BrightCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2,266,168,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,3,266,166,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,4,266,164,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,5,266,162,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5


,ID,COL,ROW,NAME,SPOT_ID,CONTROL_TYPE,REFSEQ,GB_ACC,GENE,GENE_SYMBOL,...,ENSEMBL_ID,TIGR_ID,ACCESSION_STRING,CHROMOSOMAL_LOCATION,CYTOBAND,DESCRIPTION,GO_ID,SEQUENCE,SPOT_ID.1,ORDER
45215,45216,1,9,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45216
45216,45217,1,7,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45217
45217,45218,1,5,DarkCorner,DarkCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45218
45218,45219,1,3,GE_BrightCorner,GE_BrightCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45219
45219,45220,1,1,GE_BrightCorner,GE_BrightCorner,pos,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45220


In [21]:
# Select the annotation columns we need
annotation = gpl.table[
    ["ID", "GENE_SYMBOL", "GENE_NAME", "SPOT_ID"]
].copy()

# Convert IDs to the same type before matching
annotation["ID"] = annotation["ID"].astype(str)
X_probe_ids = X.columns.astype(str)

print("First expression probe IDs:", X_probe_ids[:10].tolist())
print("First annotation IDs:", annotation["ID"].head(10).tolist())

# Check how many expression probes have annotation entries
annotation_ids = set(annotation["ID"])
matched = sum(probe_id in annotation_ids for probe_id in X_probe_ids)

print(f"Expression probes matched: {matched:,} / {len(X_probe_ids):,}")

First expression probe IDs: ['12', '13', '14', '15', '16', '17', '18', '19', '20', '22']
First annotation IDs: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
Expression probes matched: 30,635 / 30,635


In [22]:
# Keep the annotation fields we need
annotation = gpl.table[
    ["ID", "GENE_SYMBOL", "GENE_NAME", "SPOT_ID"]
].copy()

annotation["ID"] = annotation["ID"].astype(str)
annotation = annotation.drop_duplicates(subset="ID")

# Create a lookup from probe ID to gene symbol
symbol_lookup = annotation.set_index("ID")["GENE_SYMBOL"]

# Give each feature a readable name while preserving its probe ID
new_columns = []

for probe_id in X.columns.astype(str):
    symbol = symbol_lookup.get(probe_id)

    if pd.notna(symbol) and str(symbol).strip():
        feature_name = f"{str(symbol).strip()} [probe {probe_id}]"
    else:
        feature_name = f"Unknown [probe {probe_id}]"

    new_columns.append(feature_name)

# Create an annotated copy; leave the original X unchanged
X_annotated = X.copy()
X_annotated.columns = new_columns

print("Original matrix shape:", X.shape)
print("Annotated matrix shape:", X_annotated.shape)

display(pd.DataFrame({
    "probe_id": X.columns.astype(str)[:15],
    "annotated_feature": X_annotated.columns[:15]
}))

Original matrix shape: (88, 30635)
Annotated matrix shape: (88, 30635)


,probe_id,annotated_feature
0,12,APOBEC3B [probe 12]
1,13,Unknown [probe 13]
2,14,ATP11B [probe 14]
3,15,LOC100132006 [probe 15]
4,16,DNAJA1 [probe 16]
5,17,Unknown [probe 17]
6,18,EHMT2 [probe 18]
7,19,RPL23 [probe 19]
8,20,Unknown [probe 20]
9,22,RPS13 [probe 22]


In [23]:
# Count probes with and without a usable gene symbol
has_symbol = (
    symbol_lookup.reindex(X.columns.astype(str))
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

annotated_count = int(has_symbol.sum())
unannotated_count = int((~has_symbol).sum())

print(f"Probes with gene symbols: {annotated_count:,}")
print(f"Probes without gene symbols: {unannotated_count:,}")
print(f"Percentage annotated: {annotated_count / len(X.columns) * 100:.2f}%")

Probes with gene symbols: 24,238
Probes without gene symbols: 6,397
Percentage annotated: 79.12%


## Check for missing expression values

In [24]:
import numpy as np

# Check for missing and infinite expression values
missing_count = int(X.isna().sum().sum())
infinite_count = int(np.isinf(X.to_numpy()).sum())

print("Expression matrix shape:", X.shape)
print(f"Missing expression values: {missing_count:,}")
print(f"Infinite expression values: {infinite_count:,}")
print(f"Total expression values: {X.size:,}")

# Check whether any probes have missing values
probes_with_missing = int(X.isna().any(axis=0).sum())
print(f"Probes containing missing values: {probes_with_missing:,}")

Expression matrix shape: (88, 30635)
Missing expression values: 33,916
Infinite expression values: 0
Total expression values: 2,695,880
Probes containing missing values: 11,272


## Check distribution of missing values

In [25]:
# Missing values per sample
missing_per_sample = X.isna().sum(axis=1)

print("Missing values per sample:")
display(
    missing_per_sample.rename("missing_values")
    .sort_values(ascending=False)
    .to_frame()
)

print("Samples with at least one missing value:",
      int((missing_per_sample > 0).sum()))

# Summary of missingness across samples and probes
print("\nMissing values per sample — summary:")
display(missing_per_sample.describe())

print("\nProbes with the most missing values:")
display(
    X.isna().sum(axis=0)
    .sort_values(ascending=False)
    .head(10)
    .rename("missing_values")
    .to_frame()
)

Missing values per sample:


,missing_values
sample_id,
GSM1521689,3472
GSM1521658,3121
GSM1521659,2104
GSM1521688,2077
GSM1521698,1669
...,...
GSM1521675,39
GSM1521636,38
GSM1521691,36


Samples with at least one missing value: 88

Missing values per sample — summary:


count      88.000000
mean      385.409091
std       587.824484
min        16.000000
25%       116.000000
50%       203.000000
75%       355.750000
max      3472.000000
dtype: float64


Probes with the most missing values:


,missing_values
ID_REF,
12,7
20,7
16660,7
45207,7
41,7
2934,7
2931,7
2930,7
2875,7


## Check if missing values are evenly distributed by diagnosis

In [26]:
# Join each sample's missing-value count to its diagnosis
missing_summary = missing_per_sample.rename(
    "missing_values"
).to_frame()

# Add the diagnosis from the cleaned metadata
diagnosis_lookup = metadata_clean.set_index("sample_id")["disease state"]

missing_summary["disease_state"] = diagnosis_lookup.reindex(
    missing_summary.index
)

# Calculate missingness as a percentage of all probes
missing_summary["missing_percent"] = (
    missing_summary["missing_values"] / X.shape[1] * 100
)

display(
    missing_summary.sort_values(
        "missing_values", ascending=False
    ).head(15)
)

print("\nMissingness by diagnosis:")
display(
    missing_summary.groupby("disease_state")["missing_percent"]
    .agg(["count", "mean", "median", "max"])
    .round(2)
)

,missing_values,disease_state,missing_percent
sample_id,,,
GSM1521689,3472,schizophrenia,11.333442
GSM1521658,3121,healthy control,10.187694
GSM1521659,2104,healthy control,6.867961
GSM1521688,2077,schizophrenia,6.779827
GSM1521698,1669,schizophrenia,5.448017
GSM1521661,1142,healthy control,3.727762
GSM1521633,1115,bipolar disorder,3.639628
GSM1521700,872,schizophrenia,2.846417
GSM1521631,733,bipolar disorder,2.392688



Missingness by diagnosis:


,count,mean,median,max
disease_state,,,,
bipolar disorder,29,0.91,0.81,3.64
healthy control,30,1.32,0.60,10.19
schizophrenia,29,1.54,0.69,11.33


## Other basic checks

In [27]:
# Basic data-quality checks
print("Duplicate sample IDs:", int(X.index.duplicated().sum()))
print("Duplicate probe IDs:", int(X.columns.duplicated().sum()))
print("Samples missing diagnosis:",
      int(metadata_clean["disease state"].isna().sum()))

print("\nDiagnosis counts:")
display(metadata_clean["disease state"].value_counts(dropna=False))

print("\nExpression values summary:")
display(X.describe().loc[["mean", "std", "min", "max"]].T.head(10))

Duplicate sample IDs: 0
Duplicate probe IDs: 0
Samples missing diagnosis: 0

Diagnosis counts:


disease state
healthy control     30
bipolar disorder    29
schizophrenia       29
Name: count, dtype: int64


Expression values summary:


,mean,std,min,max
ID_REF,,,,
12,1.228132,0.269116,0.808228,1.915723
13,1.424626,0.281483,0.816715,2.155397
14,2.570781,0.106976,2.291715,2.847255
15,1.262176,0.329300,0.806051,2.918675
16,3.775471,0.155014,3.338854,4.150209
17,3.542537,0.172144,2.863407,3.963771
18,2.314156,0.266143,1.452029,2.927364
19,4.005640,0.421681,2.274906,4.358337
20,1.390734,0.255926,0.911690,1.998326


In [31]:

# Check missingness in selected metadata fields
metadata_fields = ["age", "population", "tissue", "age of onset"]

missing_summary = pd.DataFrame({
    "missing_count": metadata_clean[metadata_fields].isna().sum(),
    "missing_percent": (
        metadata_clean[metadata_fields].isna().mean() * 100
    ).round(2),
    "non_missing_count": metadata_clean[metadata_fields].notna().sum()
})

print("Missingness in selected metadata fields:")
display(missing_summary)

# Check unique tissue values
print("\nUnique values of tissue:")
print("Number of unique non-missing values:",
      metadata_clean["tissue"].nunique())
display(
    metadata_clean["tissue"]
    .value_counts(dropna=False)
    .rename_axis("tissue")
    .to_frame("sample_count")
)

# Check unique population values
print("\nUnique values of population:")
print("Number of unique non-missing values:",
      metadata_clean["population"].nunique())
display(
    metadata_clean["population"]
    .value_counts(dropna=False)
    .rename_axis("population")
    .to_frame("sample_count")
)

Missingness in selected metadata fields:


,missing_count,missing_percent,non_missing_count
age,0,0.00,88
population,0,0.00,88
tissue,0,0.00,88
age of onset,31,35.23,57



Unique values of tissue:
Number of unique non-missing values: 1


,sample_count
tissue,
brain (frontal cortex),88



Unique values of population:
Number of unique non-missing values: 3


,sample_count
population,
white,86
Native American,1
Hispanic,1


## Check if missing values for age of onset are only for controls

In [35]:
onset_check = pd.DataFrame({
    "disease_state": metadata_clean["disease state"],
    "age_of_onset": metadata_clean["age of onset"]
})

onset_check["age_of_onset_missing"] = (
    onset_check["age_of_onset"].isna()
)

print("Age-of-onset missingness by diagnosis:")
display(
    onset_check.groupby("disease_state")["age_of_onset_missing"]
    .agg(
        total_samples="count",
        missing_onset="sum"
    )
    .assign(
        non_missing_onset=lambda df:
            df["total_samples"] - df["missing_onset"],
        missing_percent=lambda df:
            (df["missing_onset"] / df["total_samples"] * 100).round(2)
    )
)

# List samples with missing age of onset
print("\nSamples with missing age of onset:")
display(
    onset_check[onset_check["age_of_onset_missing"]]
    [["disease_state", "age_of_onset"]]
)


# Find healthy controls with a recorded age of onset
control_with_onset = metadata_clean[
    (metadata_clean["disease state"] == "healthy control")
    & (metadata_clean["age of onset"].notna())
]

print("Number of healthy controls with recorded age of onset:", len(control_with_onset))

display(control_with_onset)

Age-of-onset missingness by diagnosis:


,total_samples,missing_onset,non_missing_onset,missing_percent
disease_state,,,,
bipolar disorder,29,2,27,6.90
healthy control,30,29,1,96.67
schizophrenia,29,0,29,0.00



Samples with missing age of onset:


,disease_state,age_of_onset
11,bipolar disorder,NaN
20,bipolar disorder,NaN
29,healthy control,NaN
30,healthy control,NaN
31,healthy control,NaN
32,healthy control,NaN
33,healthy control,NaN
34,healthy control,NaN
35,healthy control,NaN
36,healthy control,NaN


Number of healthy controls with recorded age of onset: 1


,sample_id,title,tissue,disease state,age,population,dsm-iv,age of onset,gender
49,GSM1521674,CON21,brain (frontal cortex),healthy control,19 yr,white,296.73,17 yr,NaN


## Check what's going on with GSM1521674

In [36]:
sample_id = "GSM1521674"
sample = gse.gsms[sample_id]

print("GEO sample ID:", sample_id)
print("=" * 70)

for field, values in sample.metadata.items():
    print(f"\n{field}:")
    for value in values:
        print(f"  {value}")

GEO sample ID: GSM1521674

title:
  CON21

geo_accession:
  GSM1521674

status:
  Public on Oct 11 2014

submission_date:
  Oct 08 2014

last_update_date:
  Oct 14 2014

type:
  RNA

channel_count:
  2

source_name_ch1:
  frontal cortex, healthy control

organism_ch1:
  Homo sapiens

taxid_ch1:
  9606

characteristics_ch1:
  tissue: brain (frontal cortex)
  disease state: healthy control
  age: 19 yr
  population: white
  dsm-iv: 296.73
  age of onset: 17 yr

treatment_protocol_ch1:
  N/A

growth_protocol_ch1:
  Frozen tissue.

molecule_ch1:
  total RNA

extract_protocol_ch1:
  Total RNA was isolated using TRIzol (Sigma) according to the manufacturer's instructions.
  RNA concentration was determined by spectrophotometry (NanoDrop, Thermo Scientific, US), and integrity was assessed using the 2100 Bioanalyzer (Agilent Technologies, Waldbronn, Germany).

label_ch1:
  Cy3

label_protocol_ch1:
  Agilent Technologies' manufacturer procedures.

source_name_ch2:
  Reference

organism_ch2:
  H

## Exploratory Data Analysis — Summary

### 1. Dataset Overview

- **Source:** GEO accession GSE62191
- **Platform:** GPL4133
- **Original sample count:** 88 human frontal-cortex samples
- **Features:** 30,635 microarray probes
- **Target:** `disease state`

### 2. Class Distribution

The original dataset contains 30 healthy controls, 29 bipolar disorder samples and 29 schizophrenia samples. The classes are nearly balanced, but the sample size is small relative to the number of features, increasing the risk of overfitting.

### 3. Probe Annotation

Of the 30,635 probes, 24,238 (79.12%) have usable gene symbols, while 6,397 (20.88%) do not. Original probe IDs were retained for traceability, and probes mapping to the same gene were not merged.

### 4. Metadata Quality and Sample Exclusion

Sample GSM1521674 (CON21) was labelled as a healthy control but had an age of 19, an age of onset of 17, and a DSM-IV field value of `296.73` so will be excluded from both the expression matrix and metadata. The modelling dataset will therefore contain 87 samples 

Other metadata findings:

- **Age:** Complete in the original dataset.
- **Population:** 86 samples are recorded as white, one as Native American and one as Hispanic.
- **Tissue:** All samples are recorded as brain (frontal cortex).
- **Age of onset:** Missing for 31 of the original 88 samples (35.23%), predominantly among healthy controls.
- **Gender:** Missing for 87 of the original samples and will not be used as a predictor.

### 5. Gene-Expression Data Quality

The original expression matrix contains 33,916 missing values (approximately 1.26% of all measurements), spread across all samples. A total of 11,272 probes contain at least one missing value.

Mean missingness differs by diagnosis: 0.91% for bipolar disorder, 1.32% for healthy controls and 1.54% for schizophrenia.

Missingness statistics will be recalculated after excluding GSM1521674.

### 6. Data Integrity

The initial checks found no duplicate sample IDs, duplicate probe IDs or missing diagnosis labels. All expression probes matched records in the platform annotation.

Following the exclusion, sample identifiers and row counts will be checked to ensure that the expression matrix and metadata remain aligned.

### 7. Conclusion and Next Steps

The main limitations are the small sample size relative to the number of features, missing expression measurements, and incomplete metadata.

The next notebook, `02_preprocessing_and_baseline.ipynb`, will:

1. Load the saved expression matrix and metadata.
2. Exclude GSM1521674 from both and verify alignment.
3. Prepare the diagnosis labels.
4. Create a reproducible stratified train/test split.
5. Impute missing values using training data only.
6. Evaluate a `DummyClassifier` baseline before testing more complex models.